# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fasiuu/FLYRANK_TASK1/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
# Setup — shared feature builder from this repo + DuckDB connection to the hosted release.
# Token: Colab Secret HF_TOKEN (preferred) or a getpass prompt. Never paste a token into a cell.
%pip -q install duckdb
import os, sys, subprocess, json, warnings
warnings.filterwarnings('ignore')
if not os.path.exists('FLYRANK_TASK1'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/Fasiuu/FLYRANK_TASK1.git'], check=True)
sys.path.insert(0, 'FLYRANK_TASK1/work/scripts')
import numpy as np, pandas as pd
import flyrank_features as ff
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
np.random.seed(ff.SEED)
con = ff.connect()
os.makedirs('work/outputs', exist_ok=True)
print('connected; seed =', ff.SEED)

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Distributions first. Impressions are very heavy-tailed: the mean is several times the median, so a few big pages would dominate any raw correlation. Every test below uses buckets with visible counts, and I check each one on both development snapshots (April and May outcomes). A signal that only shows up in one month doesn't count.

In [ ]:
# Build the per-item panel for the two development decision dates (one DuckDB scan, ~5-8 min).
import time
t0 = time.time()
panel = ff.build_panel(con, ff.DEV_DATES)
panel['decision_date'] = pd.to_datetime(panel['decision_date']).dt.strftime('%Y-%m-%d')
m = ff.model_frame(panel)
print(f'{len(panel):,} item-date rows built in {time.time()-t0:.0f}s; {len(m):,} pass the population checks')
cols = ['imp_last30', 'clicks_last30', 'pos_last30', 'content_age_days', 'spike_share', 'imp_momentum']
print(m[cols].describe(percentiles=[.25, .5, .75, .9, .99]).T.round(2).to_string())
print('\nimpressions mean/median ratio:', round(m.imp_last30.mean() / m.imp_last30.median(), 1), '-> heavy tail: tests use buckets and log scales, never raw Pearson')

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Each test uses the site-relative label: did the page fall more than 20% behind its own site?

**Test 1: "older pages decline more" (FlyRank's decay-after-270-days story). Verdict: OPPOSITE.** Pages aged 90-180 days fell behind about 53-54% of the time; pages over a year old only 36-42%. Measured against their own site, old pages that still have 200+ impressions are the *steadier* ones. My read is that old pages which survive with real traffic have already proven themselves. This is different from the paper's health-score comparison, which wasn't site-relative.

**Test 2: "a page that's already sliding keeps sliding". Verdict: CONFIRMED.** Pages that fell more than 40% last month fell behind their site 74-83% of the time, against 24-40% for pages that rose. Same direction in both months, with large buckets (thousands of rows each).

**Test 3: "a slipping average position predicts a decline". Verdict: MIXED.** Pages that slipped more than 3 places do fall behind somewhat more often (52-58% vs 45-46% for steady), but pages that *improved* more than 3 places sit at 37-50%, so the pattern isn't clean and the gap is small.

In [ ]:
def rate_by(col, bins, labels, df=m):
    b = pd.cut(df[col], bins=bins, labels=labels, include_lowest=True)
    return df.groupby([b, 'decision_date'], observed=True).declined.agg(['mean', 'size']).unstack('decision_date').round(3)
print('TEST 1 - "older pages decline more" (FlyRank paper: decay after ~270 days)')
print(rate_by('content_age_days', [90, 180, 270, 365, 10000], ['90-180', '181-270', '271-365', '365+']).to_string())
print('\nTEST 2 - "a page that is already sliding keeps sliding" (momentum)')
print(rate_by('imp_momentum', [-20, -0.5, -0.15, 0.15, 0.5, 20], ['fell >40%', 'fell 14-40%', 'flat', 'rose 16-65%', 'rose >65%']).to_string())
print('\nTEST 3 - "a slipping average position predicts a decline"')
print(rate_by('pos_change', [-1000, -3, -1, 1, 3, 1000], ['improved >3', 'improved 1-3', 'steady', 'slipped 1-3', 'slipped >3']).to_string())

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

**Flag-linked test: `page_one_decay_risk`** (average position ≤ 10 and age ≥ 180 days). The starter pipeline uses this as a reason code, and FlyRank's paper says "protect page-one assets". **Verdict: FALSE for this label.** Old page-one pages fall behind their site at almost exactly the same rate as everything else (≈49% vs ≈48-49% in both months). The flag doesn't separate risk.

The position tiers are more interesting: top-3 pages fell behind *more* often in May (74%). That looks like regression to the mean: pages at the very top have more room to lose.

In [ ]:
# Flag-linked test: the starter's page_one_decay_risk reason code (avg_position <= 10 and age >= 180)
# and FlyRank's "protect page-one assets" advice assume old page-one pages are the ones at risk.
m['page_one_old'] = (m.pos_last30 <= 10) & (m.content_age_days >= 180)
print(m.groupby(['page_one_old', 'decision_date']).declined.agg(['mean', 'size']).unstack('decision_date').round(3).to_string())
print('\nby position tier:')
print(rate_by('pos_last30', [0, 3, 10, 20, 50, 1000], ['top3', '4-10', '11-20', '21-50', '50+']).to_string())

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

What a content team should take from this:

1. **Watch momentum and one-day spikes, not age.** A page whose last month was carried by one spike day falls behind its site most of the time (75-90% for spike shares above 25%), so check where the spike came from before reading the month as normal.
2. **Don't schedule refreshes by age alone.** In this data, older pages that still earn traffic are the steadier ones relative to their site.
3. **Page-one status on its own isn't a risk signal.** Use it as context, not as a trigger.

In [ ]:
print('the strongest single pattern - one-day spikes revert:')
print(rate_by('spike_share', [0, .1, .25, .5, 1.0001], ['<10%', '10-25%', '25-50%', '>50%']).to_string())

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.